In [26]:
import pandas as pd
import numpy as np
import lightgbm as lgb
model = lgb.Booster(model_file='../submissions/lgb_model.txt') #kaydettiğimiz modeli yükle 

test = pd.read_csv('../data/test.csv', parse_dates=['date'])
stores = pd.read_csv('../data/stores.csv')
oil = pd.read_csv('../data/oil.csv', parse_dates=['date'])
holidays = pd.read_csv('../data/holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv('../data/transactions.csv', parse_dates=['date'])
train = pd.read_csv('../data/train.csv', parse_dates=['date'])

oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill() #oil'deki boş satırları doldur

print(f"Test boyutu: {test.shape} \n {test.head()}")



Test boyutu: (28512, 5) 
         id       date  store_nbr      family  onpromotion
0  3000888 2017-08-16          1  AUTOMOTIVE            0
1  3000889 2017-08-16          1   BABY CARE            0
2  3000890 2017-08-16          1      BEAUTY            2
3  3000891 2017-08-16          1   BEVERAGES           20
4  3000892 2017-08-16          1       BOOKS            0


In [27]:
# Test verisine featureları ekle

# Tatil verisini hazırla
ulusal_tatiller = holidays[holidays['locale'] == 'National'][['date','type']].copy()
ulusal_tatiller = ulusal_tatiller.rename(columns={'type': 'tatil_tipi'})
ulusal_tatiller = ulusal_tatiller.drop_duplicates(subset='date')

# Test ile birleştir
test_df = test.merge(stores, on='store_nbr', how='left')
test_df = test_df.merge(oil, on='date', how='left')
test_df = test_df.merge(transactions, on=['date','store_nbr'], how='left')
test_df = test_df.merge(ulusal_tatiller, on='date', how='left')

test_df['dcoilwtico']   = test_df['dcoilwtico'].ffill().bfill()
test_df['transactions'] = test_df['transactions'].fillna(0)
test_df['tatil_tipi']   = test_df['tatil_tipi'].fillna('Normal')

print(f"Test birleştirildi: {test_df.shape}")

Test birleştirildi: (28512, 12)


In [28]:
#Tarih değişkenleri ve Encoding
from sklearn.preprocessing import LabelEncoder

test_df['yil']           = test_df['date'].dt.year
test_df['ay']            = test_df['date'].dt.month
test_df['gun']           = test_df['date'].dt.day
test_df['haftanin_gunu'] = test_df['date'].dt.dayofweek
test_df['hafta_sonu']    = (test_df['haftanin_gunu'] >= 5).astype(int)
test_df['ayin_haftasi']  = test_df['date'].dt.isocalendar().week.astype(int)
test_df['promosyon_var'] = (test_df['onpromotion'] > 0).astype(int)

# Encoding — train ile aynı sıralamayı kullanmak için (trainden fit edip teste uygula)
le = LabelEncoder() # 

# Train + test birleştirerek fit et(tutarlı olması için), Sadece test'e fit edersen train'deki bazı kategoriler eksik kalır
for col, enc_col in [('family','family_enc'), ('city','city_enc'),
                      ('state','state_enc'), ('type','type_enc'),
                      ('tatil_tipi','tatil_enc')]:
    le.fit(pd.concat([train.get(col, test_df[col]),
                      test_df[col]]).astype(str).unique())
    test_df[enc_col] = le.transform(test_df[col].astype(str))

print(f"Encoding tamamlandı \n {test_df.head()}")

Encoding tamamlandı 
         id       date  store_nbr      family  onpromotion   city      state  \
0  3000888 2017-08-16          1  AUTOMOTIVE            0  Quito  Pichincha   
1  3000889 2017-08-16          1   BABY CARE            0  Quito  Pichincha   
2  3000890 2017-08-16          1      BEAUTY            2  Quito  Pichincha   
3  3000891 2017-08-16          1   BEVERAGES           20  Quito  Pichincha   
4  3000892 2017-08-16          1       BOOKS            0  Quito  Pichincha   

  type  cluster  dcoilwtico  ...  gun haftanin_gunu  hafta_sonu  ayin_haftasi  \
0    D       13        46.8  ...   16             2           0            33   
1    D       13        46.8  ...   16             2           0            33   
2    D       13        46.8  ...   16             2           0            33   
3    D       13        46.8  ...   16             2           0            33   
4    D       13        46.8  ...   16             2           0            33   

   promosyon_var

In [29]:
#Test için LAG değişkenleri 
# Test tarihleri 2017-08-16 -> 2017-08-31 (09 ile 24. günler arası trainden gelir )

# Train'in son kısmını al — lag için gerekli geçmiş
son_train = train.sort_values(['store_nbr','family','date'])

# Her mağaza-ürün için son 30 günlük satışı al
lag_ref = son_train.groupby(['store_nbr','family']).tail(30)

def get_lag(test_df, lag_gun):
    # Test tarihinden lag_gun gün önce train'de ne vardı?
    lag_date = test_df[['store_nbr','family','date']].copy()
    lag_date['lag_date'] = lag_date['date'] - pd.Timedelta(days=lag_gun)

    lag_values = son_train[['store_nbr','family','date','sales']].copy()
    lag_values = lag_values.rename(columns={
        'date': 'lag_date',
        'sales': f'lag_{lag_gun}'
    })

    return lag_date.merge(lag_values,
                          on=['store_nbr','family','lag_date'],
                          how='left')[f'lag_{lag_gun}']


test_df['lag_3']  = get_lag(test_df, 3)
test_df['lag_7']  = get_lag(test_df, 7)
test_df['lag_10']  = get_lag(test_df, 10)
test_df['lag_14']  = get_lag(test_df, 14)
test_df['lag_21']  = get_lag(test_df, 21)
test_df['lag_24'] = get_lag(test_df, 24)
test_df['lag_28'] = get_lag(test_df, 28)

# Rolling ortalama
for gun in [3,7,10,14,21,24,28]:
    test_df[f'rolling_{gun}'] = get_lag(test_df, gun)


# Kalan eksikleri doldur
for col in ['lag_3','lag_7','lag_10','lag_14','lag_21','lag_24','lag_28','rolling_3','rolling_7','rolling_10','rolling_14','rolling_21','lag_24','rolling_28']:
    test_df[col] = test_df[col].fillna(0)

print(f"Lag ve rolling değişkenleri eklendi: \n {test_df[['store_nbr','date','family','lag_3','rolling_3']].head()}")


Lag ve rolling değişkenleri eklendi: 
    store_nbr       date      family  lag_3  rolling_3
0          1 2017-08-16  AUTOMOTIVE    1.0        1.0
1          1 2017-08-16   BABY CARE    0.0        0.0
2          1 2017-08-16      BEAUTY    1.0        1.0
3          1 2017-08-16   BEVERAGES  803.0      803.0
4          1 2017-08-16       BOOKS    0.0        0.0


In [31]:
#Tahmin ve submission.csv

features = [
    # Mağaza bilgileri
    'store_nbr', 'cluster',
    'type_enc', 'city_enc', 'state_enc',

    # Ürün bilgileri
    'family_enc',

    # Promosyon
    'promosyon_var', 'onpromotion',

    # Dışsal faktörler
    'dcoilwtico', 'transactions', 'tatil_enc',

    # Tarih değişkenleri
    'yil', 'ay', 'gun', 'haftanin_gunu',
    'hafta_sonu', 'ayin_haftasi',

    # Lag değişkenleri
    'lag_3', 'lag_7', 'lag_10','lag_14','lag_21','lag_24','lag_28',

    # Rolling ortalamalar
    'rolling_3', 'rolling_7','rolling_10','rolling_14','rolling_21','rolling_24','rolling_28'
]

#tahmin yap
X_test = test_df[features]
y_pred_log = model.predict(X_test)

y_pred = np.expm1(y_pred_log) #logdan geri çevir
y_pred = np.maximum(y_pred , 0) #negatif değer olamaz 0 yap 

submission = pd.DataFrame({
    'id' : test_df['id'],
    'sales' : y_pred
})

submission.to_csv('../submissions/submission.csv', index=False)
print(f"submission.csv oluşturuldu \n {submission.head()}")


submission.csv oluşturuldu 
         id      sales
0  3000888   0.401372
1  3000889   0.015831
2  3000890   1.546055
3  3000891  13.134265
4  3000892   0.016908
